# Estudio completo de susceptibilidad a incendios en un solo Colab

**Este es el cuaderno principal.** Contiene datos de entrada, auditoría, modelos,
validación geográfica y temporal, ablaciones, figuras, explicabilidad y orientación
para escribir el paper. No necesita resultados ni ZIP de otros cuadernos.

### Cómo usarlo

1. Ejecuta las celdas en orden, o usa **Entorno de ejecución → Ejecutar todas**.
2. Autoriza Drive si quieres conservar los datos y avances entre sesiones.
3. Carga una sola vez `dataset_modelo_incendios_sierra_2019_2025.csv`.
4. Espera la ejecución y revisa las figuras y la guía del manuscrito al final.

Los datos obligatorios ya contienen las pseudoausencias y sus covariables:
**no se vuelven a generar**. El CSV correcto está en tu equipo, dentro de
`INCENDIOS_BASE_DATOS/BASE DE DATOS`. No lo sustituyas por la base antigua
`dataset_entrenamiento_sierra_realista.csv`.

El modo **COMPLETO** ejecuta el protocolo configurado; puede tardar varias horas
en CPU. El modo RAPIDO sirve para comprobar el flujo y marca sus resultados como
piloto. El cuaderno guarda y reutiliza etapas compatibles con los mismos datos,
código, bibliotecas y parámetros; no promete recuperar una etapa interrumpida a mitad de ajuste.

### Recorrido del estudio

Configuración → entrada única de datos → auditoría → particiones → modelos →
ablaciones → figuras y SHAP → informe orientativo para el paper → guardado.

La reproducción comienza en el CSV trazable. No se afirma haber reconstruido
la extracción GEE: la diferencia entre la bitácora y el script aportado sigue
pendiente de reconciliar. Tampoco se inventan campañas nuevas de pseudoausencias
ni superficies cartográficas que no estén respaldadas por predictores espaciales.


## A. Configuración única

Los valores siguientes dejan preparado el análisis completo. No necesitas
modificar rutas de otros cuadernos. Para una prueba técnica puedes cambiar
`MODO` a `RAPIDO`. Las etapas completadas se guardan por versión.


In [ ]:
# Controles principales del único flujo de trabajo.
MODO='COMPLETO'                 # COMPLETO para el estudio; RAPIDO solo para comprobar.
GUARDAR_EN_DRIVE=True
FORZAR_CARGA_DATOS=False
RECALCULAR_ETAPAS=False
EJECUTAR_ABLACIONES=True
EJECUTAR_SHAP=True
ARCHIVO_PRINCIPAL='dataset_modelo_incendios_sierra_2019_2025.csv'

# Información que debes confirmar antes de cerrar el manuscrito.
FUENTE_INSTITUCIONAL=''         # Institución y nombre oficial de la base.
VERSION_INVENTARIO=''
ESTADO_EXTRACCION='Pendiente: reconciliar ventanas, máscara y muestreo entre bitácora y script GEE'

# Identidad de la implementación: se registra junto a los resultados.
NOTEBOOK_TEMPLATE_SHA='f63d71810fe2db0d9f6918b3b4cd61e4011621a44eef5e590c6cd91864527a53'
NOTEBOOK_IMPLEMENTATION_SHA=NOTEBOOK_TEMPLATE_SHA


## B. Preparar entorno y almacenamiento

Se instalan solo dependencias que falten. Los datos y salidas quedan en tu
carpeta de Drive `Susceptibilidad_Incendios_Estudio`; no se publican en GitHub.
Si desactivas Drive, el almacenamiento será temporal y tendrás que descargarlo
antes de que termine la sesión.


In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path
from IPython.display import display,Image,Markdown
DEPENDENCIAS={'numpy':'numpy','pandas':'pandas','scipy':'scipy','sklearn':'scikit-learn',
              'matplotlib':'matplotlib','pyproj':'pyproj','torch':'torch','shap':'shap'}
faltantes=[p for m,p in DEPENDENCIAS.items() if importlib.util.find_spec(m) is None]
if faltantes:subprocess.run([sys.executable,'-m','pip','install',*faltantes],check=True)
if GUARDAR_EN_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ALMACEN=Path('/content/drive/MyDrive/Susceptibilidad_Incendios_Estudio')
else:
    ALMACEN=Path('/content/Susceptibilidad_Incendios_Estudio')
DATOS=ALMACEN/'datos'
DATOS.mkdir(parents=True,exist_ok=True)
print('Carpeta del estudio:',ALMACEN)


## C. Código completo, visible y comentado

Las siguientes celdas definen los procedimientos que usa este mismo cuaderno.
Contienen el código real de auditoría, particiones, modelos, métricas y figuras.
No se clonan repositorios ni se ejecutan scripts descargados en segundo plano.
Las funciones se llaman en las etapas D–G. Puedes usar el índice lateral de
Colab para desplazarte entre las secciones.


### Auditoría de estructura, emparejamiento y tablas fuente

#### Bibliotecas y reglas del protocolo documentado

In [ ]:
from pathlib import Path
import hashlib
import json
import unicodedata
import numpy as np
import pandas as pd
from pyproj import Geod
from sklearn.neighbors import BallTree

NUMERIC_FEATURES=['elevacion','pendiente','orientacion','temperatura_C','precipitacion_mm','NDVI','EVI','NBR']
SIERRA={'azuay','bolivar','canar','carchi','chimborazo','cotopaxi','imbabura','loja','pichincha','tungurahua'}

def province_name(value):
    text=''.join(c for c in unicodedata.normalize('NFKD',str(value).lower()) if not unicodedata.combining(c)).strip(" '")
    return {'bolvar':'bolivar','caar':'canar'}.get(text,text)



#### Controles estructurales y emparejamiento

In [ ]:
def audit_dataset(dataset):
    df=pd.read_csv(dataset)
    required=NUMERIC_FEATURES+['id_registro','date','provincia','latitud','longitud','clase']
    if set(required)-set(df):
        raise ValueError('Esquema distinto al de la base trazable.')
    dates=pd.to_datetime(df.date,errors='coerce')
    result=dict(file=Path(dataset).name,sha256=hashlib.sha256(Path(dataset).read_bytes()).hexdigest(),rows=len(df),
        nulls=df[required].isna().sum().astype(int).to_dict(),invalid_dates=int(dates.isna().sum()),
        nonfinite=int((~np.isfinite(df[NUMERIC_FEATURES+['latitud','longitud']].to_numpy(dtype=float))).sum()),
        duplicate_ids=int(df.id_registro.duplicated().sum()),
        duplicate_spacetime=int(df.duplicated(['clase','date','latitud','longitud']).sum()),
        recurrent_locations=int(df[['latitud','longitud']].duplicated().sum()),
        date_min=str(dates.min().date()),date_max=str(dates.max().date()),
        classes={str(k):int(v) for k,v in df.clase.value_counts().items()})
    df['pair_id']=df.id_registro.astype(str).str.extract(r'^(?:P_|PA_)(\d+)',expand=False)
    groups=df.groupby('pair_id')
    result['pair_checks']=dict(unparseable=int(df.pair_id.isna().sum()),
        wrong_size=int((groups.size()!=2).sum()),wrong_classes=int((groups.clase.nunique()!=2).sum()),
        different_date=int((groups.date.nunique()!=1).sum()),different_province=int((groups.provincia.nunique()!=1).sum()))
    result['coordinate_range_valid']=bool(df.latitud.between(-90,90).all() and df.longitud.between(-180,180).all())
    result['spectral_range_violations']=dict(NDVI=int((~df.NDVI.between(-1,1)).sum()),NBR=int((~df.NBR.between(-1,1)).sum()))
    result['negative_precipitation']=int((df.precipitacion_mm<0).sum())
    return df,result



#### Contraste con tablas fuente y ámbito explícito de exclusión

In [ ]:
def audit_sources(df,presence_path,pseudo_path,inventory_path):
    presence=pd.read_csv(presence_path); pseudo=pd.read_csv(pseudo_path); raw=pd.read_csv(inventory_path)
    positive=df[df.clase==1].copy(); positive['id']=positive.id_registro.str.removeprefix('P_').astype(int)
    negative=df[df.clase==0].copy()
    psource=presence.drop_duplicates(['date','latitud','longitud'])
    cols=['latitud','longitud',*NUMERIC_FEATURES]
    pcheck=positive.merge(psource,on='id',suffixes=('_model','_source'),validate='one_to_one')
    acheck=negative.merge(pseudo,left_on='id_registro',right_on='id_pseudo',suffixes=('_model','_source'),validate='one_to_one')
    refs=pseudo.merge(positive[['id','date','provincia']],left_on='id_presencia_referencia',right_on='id',how='left',suffixes=('_a','_p'),validate='one_to_one')
    result=dict(source_presence_rows=len(presence),unique_presence_spacetime=len(psource),
        positive_matches=len(pcheck),negative_matches=len(acheck),
        positive_values_equal=all(np.allclose(pcheck[c+'_model'],pcheck[c+'_source']) for c in cols),
        negative_values_equal=all(np.allclose(acheck[c+'_model'],acheck[c+'_source']) for c in cols),
        missing_references=int(refs.id.isna().sum()),different_reference_date=int((refs.date_a!=refs.date_p).sum()),
        different_reference_province=int((refs.provincia_a!=refs.provincia_p).sum()),
        zero_valid_s2_presence=int((presence.n_observaciones_S2_validas<=0).sum()),
        zero_valid_s2_pseudo=int((pseudo.n_observaciones_S2_validas<=0).sum()))
    years=pd.to_datetime(raw.date,errors='coerce').dt.year
    # La bitácora define 2019–2025 y diez provincias, no todo el inventario 2010–2026.
    known=raw[years.between(2019,2025)&raw.provincia.map(province_name).isin(SIERRA)].dropna(subset=['latitude','longitude'])
    result['exclusion_scope']='Sierra administrative provinces, all known events 2019–2025, including spectrally unusable events'
    result['known_reference_rows']=len(known)
    if len(known):
        tree=BallTree(np.deg2rad(known[['latitude','longitude']].to_numpy()),metric='haversine')
        # Filtra con los cinco vecinos esféricos y refina en el elipsoide WGS84.
        # El elipsoide cambia el orden solo localmente; k=5 es un refinamiento operativo.
        _,indices=tree.query(np.deg2rad(negative[['latitud','longitud']].to_numpy()),k=min(5,len(known)))
        k=indices.shape[1]
        _,_,dist=Geod(ellps='WGS84').inv(np.repeat(negative.longitud.to_numpy(),k),np.repeat(negative.latitud.to_numpy(),k),
            known.longitude.to_numpy()[indices.ravel()],known.latitude.to_numpy()[indices.ravel()])
        distances=dist.reshape(-1,k).min(axis=1)
        result['minimum_exclusion_distance_wgs84_m']=float(distances.min())
        result['pseudo_under_2000_m']=int((distances<2000).sum())
        result['exclusion_distance_quantiles_m']={str(q):float(np.quantile(distances,q)) for q in [0,.01,.5,.99,1]}
    result['temporal_acquisition_verified']=False
    result['temporal_note']='CSV values and QC counters do not contain per-image acquisition timestamps; pre-event windows are documented but require extraction code/provenance to verify.'
    return result



#### Ejecución y salidas locales

In [ ]:
def run_audit(dataset,output,presence=None,pseudo=None,inventory=None):
    output=Path(output); output.mkdir(parents=True,exist_ok=True)
    df,result=audit_dataset(dataset)
    if all(p is not None for p in [presence,pseudo,inventory]):
        result['source_checks']=audit_sources(df,presence,pseudo,inventory)
    else:
        result['source_checks']='not_run: upload source presence, pseudoabsence and full inventory CSVs'
    df['year']=pd.to_datetime(df.date).dt.year
    for group in ['provincia','year']:
        df.groupby([group,'clase']).size().unstack(fill_value=0).to_csv(output/f'balance_{group}.csv')
    df[NUMERIC_FEATURES].describe(percentiles=[.01,.5,.99]).to_csv(output/'feature_ranges.csv')
    (output/'audit.json').write_text(json.dumps(result,indent=2,ensure_ascii=False),encoding='utf-8')
    return result


### Particiones externas, modelos y ablación de atención

#### Bibliotecas y configuración

In [ ]:
from pathlib import Path
import argparse
import copy
import hashlib
import importlib.metadata
import json
import random
import time
import numpy as np
import pandas as pd
from pyproj import Transformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, f1_score, log_loss, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import GroupShuffleSplit, LeaveOneGroupOut, StratifiedGroupKFold
from sklearn.neighbors import BallTree
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

FEATURES = ['elevacion', 'pendiente', 'orientacion', 'temperatura_C', 'precipitacion_mm', 'NDVI', 'EVI', 'NBR']
DEFAULT = dict(seed=42, block_km=20, grid_offset_km=0, buffer_km=5, n_splits=5,
    development_end=2023, temporal_years=[2024], allow_final_2025=False,
    models=['logistic', 'rf', 'histgb', 'mlp', 'mlp_matched', 'attention'],
    feature_set='complete', epochs=40, patience=8, batch_size=256,
    learning_rate=0.001, token_dim=16, head_width=64, rf_trees=300, rf_depth=15,
    seeds=[42, 137, 271], pilot=False, bootstrap_reps=500,
    schemes=['random_pairs','spatial_blocks','leave_province_out','temporal_retrospective'])



#### Esquema y trazabilidad

In [ ]:
def load_data(path):
    """Exige el esquema nuevo. No mezcla la base antigua de 25.901 filas."""
    df = pd.read_csv(path)
    required = set(FEATURES + ['id_registro', 'date', 'provincia', 'latitud', 'longitud', 'clase'])
    if required - set(df.columns):
        raise ValueError('Columnas faltantes: ' + str(sorted(required-set(df.columns))))
    if df[list(required)].isna().any().any():
        raise ValueError('Hay valores faltantes; audita el origen antes de modelar.')
    df['date'] = pd.to_datetime(df['date'], errors='raise')
    df['year'] = df.date.dt.year
    if not df.year.between(2019, 2025).all() or set(df.clase.unique()) != {0, 1}:
        raise ValueError('Periodo o clases fuera del protocolo 2019–2025.')
    if not np.isfinite(df[FEATURES + ['latitud','longitud']].to_numpy(dtype=float)).all():
        raise ValueError('Hay valores no finitos.')
    if not df.latitud.between(-90,90).all() or not df.longitud.between(-180,180).all():
        raise ValueError('Coordenadas inválidas.')
    if df.id_registro.duplicated().any() or df.duplicated(['clase','date','latitud','longitud']).any():
        raise ValueError('Hay IDs o eventos espacio-temporales duplicados.')
    # El sufijo de reemplazo de una pseudoausencia no cambia su presencia de referencia.
    df['pair_id'] = df.id_registro.astype(str).str.extract(r'^(?:P_|PA_)(\d+)', expand=False)
    if df.pair_id.isna().any():
        raise ValueError('No se puede recuperar la referencia; conserva su columna explícita.')
    grouped = df.groupby('pair_id')
    if not grouped.size().eq(2).all() or not grouped.clase.nunique().eq(2).all():
        raise ValueError('Cada referencia debe tener una presencia y una pseudoausencia.')
    if grouped.date.nunique().max() != 1 or grouped.provincia.nunique().max() != 1:
        raise ValueError('El emparejamiento temporal/provincial no coincide.')
    return df.reset_index(drop=True)


def design_matrix(df, feature_set='complete'):
    """Ocho variables físicas; orientación circular se convierte a seno y coseno."""
    names = FEATURES.copy()
    if feature_set == 'no_spectral':
        names = [n for n in names if n not in ['NDVI', 'EVI', 'NBR']]
    elif feature_set == 'no_climate':
        names = [n for n in names if n not in ['temperatura_C', 'precipitacion_mm']]
    elif feature_set not in ['complete', 'raw_aspect']:
        raise ValueError('Ablación desconocida: ' + feature_set)
    X = df[names].copy()
    if feature_set != 'raw_aspect':
        angle = np.deg2rad(X.pop('orientacion'))
        X['orientacion_sin'] = np.sin(angle)
        X['orientacion_cos'] = np.cos(angle)
    return X


def spatial_groups(df, block_km, offset_km=0):
    # Proyección métrica de área equivalente; identificadores de celda, nunca predictores.
    transformer = Transformer.from_crs('EPSG:4326', 'EPSG:6933', always_xy=True)
    x, y = transformer.transform(df.longitud.to_numpy(), df.latitud.to_numpy())
    gx, gy = np.floor((np.asarray(x)-offset_km*1000)/(block_km*1000)).astype(int), np.floor((np.asarray(y)-offset_km*1000)/(block_km*1000)).astype(int)
    return pd.Series([f'{a}:{b}' for a,b in zip(gx,gy)], index=df.index)


def nearest_m(train, test):
    if not len(train) or not len(test):
        return np.array([])
    tree = BallTree(np.deg2rad(test[['latitud','longitud']].to_numpy()), metric='haversine')
    return tree.query(np.deg2rad(train[['latitud','longitud']].to_numpy()), k=1)[0][:,0] * 6371008.8


def purge(df, train, test, buffer_km=0):
    """Retira referencias emparejadas y, cuando corresponde, vecinos del test."""
    train, test = np.asarray(train), np.asarray(test)
    train = train[~df.loc[train,'pair_id'].isin(df.loc[test,'pair_id']).to_numpy()]
    if buffer_km:
        train = train[nearest_m(df.loc[train],df.loc[test]) >= buffer_km*1000]
    if len(train)<100 or df.loc[train,'clase'].nunique()!=2 or df.loc[test,'clase'].nunique()!=2:
        raise ValueError('Partición insuficiente después de la exclusión. Revisa diseño y cobertura.')
    if set(train)&set(test) or set(df.loc[train,'pair_id'])&set(df.loc[test,'pair_id']):
        raise AssertionError('Fuga de filas o referencias entre conjuntos.')
    return train, test



#### Particiones externas independientes del ajuste de modelos

In [ ]:
def make_splits(df, config):
    """2025 permanece reservado; el piloto usa un fold por diseño."""
    cfg = config
    if 2025 in cfg['temporal_years'] and not cfg['allow_final_2025']:
        raise ValueError('2025 está reservado. Congela el protocolo antes de habilitarlo.')
    if cfg['development_end'] >= 2025 and not cfg['allow_final_2025']:
        raise ValueError('El desarrollo no puede incluir 2025 mientras permanezca reservado.')
    dev = df.index[df.year <= cfg['development_end']].to_numpy()
    groups = spatial_groups(df,cfg['block_km'],cfg.get('grid_offset_km',0))
    rows = []
    splitter = GroupShuffleSplit(n_splits=1,test_size=.2,random_state=cfg['seed'])
    tr,te = next(splitter.split(dev,df.loc[dev,'clase'],df.loc[dev,'pair_id']))
    rows.append(('random_pairs', 'diagnostic', *purge(df,dev[tr],dev[te])))
    splitter = StratifiedGroupKFold(n_splits=cfg['n_splits'],shuffle=True,random_state=cfg['seed'])
    for i,(tr,te) in enumerate(splitter.split(dev,df.loc[dev,'clase'],groups.loc[dev])):
        train,test = purge(df,dev[tr],dev[te],cfg['buffer_km'])
        rows.append(('spatial_blocks',f'fold_{i+1}',train,test))
        if cfg['pilot']:
            break
    for tr,te in LeaveOneGroupOut().split(dev,df.loc[dev,'clase'],df.loc[dev,'provincia']):
        train,test = purge(df,dev[tr],dev[te],cfg['buffer_km'])
        province = str(df.loc[test,'provincia'].iloc[0])
        rows.append(('leave_province_out',province,train,test))
        if cfg['pilot']:
            break
    for year in cfg['temporal_years']:
        start = pd.Timestamp(year=year,month=1,day=1)
        # Embargo de 90 días: evita superponer ventanas espectrales a ambos lados del corte.
        tr = df.index[df.date < start-pd.Timedelta(days=90)].to_numpy()
        te = df.index[df.year == year].to_numpy()
        train,test = purge(df,tr,te)
        rows.append(('temporal_retrospective',str(year),train,test))
    return rows


def inner_validation(df, train, scheme, config):
    """Validación interna para early stopping; nunca consulta el test externo."""
    if scheme == 'temporal_retrospective':
        last_year = int(df.loc[train,'year'].max())
        boundary = pd.Timestamp(year=last_year,month=1,day=1)
        tr = train[df.loc[train,'date'].to_numpy() < (boundary-pd.Timedelta(days=90)).to_datetime64()]
        va = train[df.loc[train,'year'].to_numpy() == last_year]
        return purge(df,tr,va)
    grouping = spatial_groups(df,config['block_km'],config.get('grid_offset_km',0)) if scheme != 'random_pairs' else df.pair_id
    split = GroupShuffleSplit(n_splits=1,test_size=.2,random_state=config['seed'])
    tr,va = next(split.split(train,df.loc[train,'clase'],grouping.loc[train]))
    return purge(df,train[tr],train[va],config['buffer_km'] if scheme!='random_pairs' else 0)



#### Modelos de referencia con hiperparámetros fijados antes del test

In [ ]:
def classic_model(name, cfg):
    if name == 'logistic':
        return make_pipeline(StandardScaler(),LogisticRegression(C=1,max_iter=1000,random_state=cfg['seed']))
    if name == 'rf':
        return RandomForestClassifier(n_estimators=cfg['rf_trees'],max_depth=cfg['rf_depth'],min_samples_leaf=2,n_jobs=-1,random_state=cfg['seed'])
    if name == 'histgb':
        # Evita la validación aleatoria interna automática del estimador.
        return HistGradientBoostingClassifier(max_iter=150,max_leaf_nodes=15,l2_regularization=1,early_stopping=False,random_state=cfg['seed'])
    raise ValueError(name)



#### Ablación de atención con representación y clasificador compartidos

In [ ]:
def neural_model(features, attention, cfg):
    import torch
    from torch import nn

    class FeatureTokens(nn.Module):
        def __init__(self):
            super().__init__()
            dim = cfg['token_dim']
            width = cfg.get('head_width',64)
            self.scale = nn.Parameter(torch.randn(features,dim)*.02)
            self.bias = nn.Parameter(torch.randn(features,dim)*.02)
            self.attention = nn.MultiheadAttention(dim,4,dropout=.1,batch_first=True) if attention else None
            self.norm = nn.LayerNorm(dim)
            self.head = nn.Sequential(nn.Flatten(),nn.Linear(features*dim,width),nn.ReLU(),nn.Dropout(.1),nn.Linear(width,32),nn.ReLU(),nn.Linear(32,1))

        def forward(self,x):
            # [tanda, variables] → [tanda, variables, dimensión del token].
            tokens = x.unsqueeze(-1)*self.scale.unsqueeze(0)+self.bias.unsqueeze(0)
            if self.attention is not None:
                mixed,_ = self.attention(tokens,tokens,tokens,need_weights=False)
                tokens = tokens+mixed
            return self.head(self.norm(tokens)).squeeze(-1)

    return FeatureTokens()


def fit_neural(X, y, df, train, test, scheme, attention, cfg, on_fitted=None):
    """Selecciona épocas internamente y vuelve a ajustar en todo el train externo."""
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
    torch.set_num_threads(min(4,torch.get_num_threads()))
    torch.use_deterministic_algorithms(True)
    device = torch.device('cpu')  # Comparabilidad local/Colab; habilitar GPU exige otra verificación.
    tr,va = inner_validation(df,train,scheme,cfg)
    scaler = StandardScaler().fit(X[tr])
    xtr = torch.tensor(scaler.transform(X[tr]),dtype=torch.float32)
    xva = torch.tensor(scaler.transform(X[va]),dtype=torch.float32,device=device)
    ytr = torch.tensor(y[tr],dtype=torch.float32)
    yva = torch.tensor(y[va],dtype=torch.float32,device=device)
    def reset_model():
        random.seed(cfg['seed']); np.random.seed(cfg['seed']); torch.manual_seed(cfg['seed'])
        return neural_model(X.shape[1],attention,cfg).to(device)
    model = reset_model()
    optimizer = torch.optim.Adam(model.parameters(),lr=cfg['learning_rate'],weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()
    loader = DataLoader(TensorDataset(xtr,ytr),batch_size=cfg['batch_size'],shuffle=True)
    history=[]; best=float('inf'); best_epoch=1; stale=0
    for epoch in range(1,cfg['epochs']+1):
        model.train(); total=0.
        for bx,by in loader:
            optimizer.zero_grad(); loss=criterion(model(bx.to(device)),by.to(device)); loss.backward(); optimizer.step()
            total+=float(loss.detach())*len(bx)
        model.eval()
        with torch.no_grad():
            val_loss=float(criterion(model(xva),yva))
        history.append(dict(epoch=epoch,training_loss=total/len(tr),validation_loss=val_loss))
        print(f"  {'attention' if attention else 'mlp'} época {epoch}: train={total/len(tr):.4f}, val={val_loss:.4f}",flush=True)
        if val_loss<best-1e-5:
            best=val_loss; best_epoch=epoch; stale=0
        else:
            stale+=1
        if stale>=cfg['patience']:
            break
    # Reajuste completo: el test externo sigue sin utilizarse.
    scaler=StandardScaler().fit(X[train]); model=reset_model()
    optimizer=torch.optim.Adam(model.parameters(),lr=cfg['learning_rate'],weight_decay=1e-4)
    dataset=TensorDataset(torch.tensor(scaler.transform(X[train]),dtype=torch.float32),torch.tensor(y[train],dtype=torch.float32))
    loader=DataLoader(dataset,batch_size=cfg['batch_size'],shuffle=True)
    for epoch in range(best_epoch):
        model.train()
        for bx,by in loader:
            optimizer.zero_grad(); loss=criterion(model(bx.to(device)),by.to(device)); loss.backward(); optimizer.step()
    model.eval()
    with torch.no_grad():
        probabilities=torch.sigmoid(model(torch.tensor(scaler.transform(X[test]),dtype=torch.float32,device=device))).cpu().numpy()
    if on_fitted is not None:
        # Permite explicar el modelo reajustado sin alterar predicciones ni métricas.
        on_fitted(model,scaler)
    return probabilities,dict(selected_epochs=best_epoch,parameters=sum(p.numel() for p in model.parameters()),history=history)



#### Métricas y registro de cada partición

In [ ]:
def metrics(y,p):
    predicted=(p>=.5).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,predicted,labels=[0,1]).ravel()
    return dict(roc_auc=roc_auc_score(y,p),average_precision=average_precision_score(y,p),
        accuracy=accuracy_score(y,predicted),balanced_accuracy=balanced_accuracy_score(y,predicted),
        precision=precision_score(y,predicted,zero_division=0),recall=recall_score(y,predicted,zero_division=0),
        f1=f1_score(y,predicted,zero_division=0),brier=brier_score_loss(y,p),log_loss=log_loss(y,p,labels=[0,1]),
        tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp))


def paired_block_bootstrap(frame,baseline='mlp',challenger='attention',reps=500,seed=42):
    """Intervalo aproximado de diferencia AUC; remuestrea bloques, no filas."""
    pivot=frame.pivot(index=['id_registro','clase','spatial_group'],columns='model',values='probability').dropna(subset=[baseline,challenger]).reset_index()
    groups=pivot.spatial_group.unique()
    if len(groups)<5:
        return dict(status='insufficient_blocks',blocks=len(groups))
    rng=np.random.default_rng(seed); estimates=[]
    locations={g:np.flatnonzero(pivot.spatial_group.to_numpy()==g) for g in groups}
    for _ in range(reps):
        sample=np.concatenate([locations[g] for g in rng.choice(groups,len(groups),replace=True)])
        subset=pivot.iloc[sample]
        if subset.clase.nunique()==2:
            estimates.append(roc_auc_score(subset.clase,subset[challenger])-roc_auc_score(subset.clase,subset[baseline]))
    if not estimates:
        return dict(status='insufficient_classes',blocks=len(groups))
    return dict(status='approximate_spatial_block_bootstrap',blocks=len(groups),replicates=len(estimates),
        delta_auc=float(roc_auc_score(pivot.clase,pivot[challenger])-roc_auc_score(pivot.clase,pivot[baseline])),
        lower=float(np.quantile(estimates,.025)),upper=float(np.quantile(estimates,.975)))


def run_validation(data_path,output_dir,config=None):
    cfg=copy.deepcopy(DEFAULT); cfg.update(config or {})
    output_dir=Path(output_dir); output_dir.mkdir(parents=True,exist_ok=True)
    df=load_data(data_path); matrix=design_matrix(df,cfg['feature_set']); X=matrix.to_numpy(dtype=np.float32); y=df.clase.to_numpy()
    groups=spatial_groups(df,cfg['block_km'],cfg.get('grid_offset_km',0))
    metadata=dict(config=cfg,data_sha256=hashlib.sha256(Path(data_path).read_bytes()).hexdigest(),features=list(matrix.columns),
        python=__import__('sys').version,versions={},status='running',interpretation='retrospective case-control discrimination; exclusion uses known fires across 2019-2025',
        code_sha256=globals().get('NOTEBOOK_IMPLEMENTATION_SHA') or (hashlib.sha256(Path(__file__).read_bytes()).hexdigest() if '__file__' in globals() else 'not_recorded'))
    for package in ['numpy','pandas','scipy','scikit-learn','torch','pyproj']:
        try: metadata['versions'][package]=importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError: pass
    (output_dir/'execution.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    scores=[]; predictions=[]; split_rows=[]; deltas=[]
    try:
        for seed in cfg['seeds']:
            current=dict(cfg,seed=seed)
            for scheme,fold,train,test in make_splits(df,current):
                if scheme not in cfg['schemes']:
                    continue
                print(f'\n{scheme}/{fold}, seed={seed}, train={len(train)}, test={len(test)}',flush=True)
                distances=nearest_m(df.loc[train],df.loc[test])
                info=dict(scheme=scheme,fold=fold,seed=seed,train=len(train),test=len(test),
                    test_prevalence=float(y[test].mean()),minimum_train_test_m=float(distances.min()),
                    shared_spatial_cells=len(set(groups.loc[train])&set(groups.loc[test])),
                    shared_provinces=len(set(df.loc[train,'provincia'])&set(df.loc[test,'provincia'])))
                split_rows.append(info)
                assignment=df.loc[np.r_[train,test],['id_registro','pair_id']].copy()
                assignment['partition']=['train']*len(train)+['test']*len(test)
                assignment.to_csv(output_dir/f'split_{scheme}_{fold}_{seed}.csv',index=False)
                for name in cfg['models']:
                    started=time.monotonic()
                    if name in ['mlp','mlp_matched','attention']:
                        neural_cfg=current.copy()
                        if name=='mlp_matched':
                            # Equipara aproximadamente el presupuesto de parámetros sin usar el test.
                            dim=current['token_dim']; width=current.get('head_width',64)
                            neural_cfg['head_width']=width+round((4*dim*dim+4*dim)/(X.shape[1]*dim+33))
                        probability,training=fit_neural(X,y,df,train,test,scheme,name=='attention',neural_cfg)
                        training['head_width']=neural_cfg.get('head_width',64)
                        history=training.pop('history')
                        pd.DataFrame(history).to_csv(output_dir/f'learning_{scheme}_{fold}_{seed}_{name}.csv',index=False)
                    else:
                        model=classic_model(name,current); model.fit(X[train],y[train]); probability=model.predict_proba(X[test])[:,1]; training={}
                    score=dict(scheme=scheme,fold=fold,seed=seed,model=name,feature_set=cfg['feature_set'],**metrics(y[test],probability),**training,seconds=time.monotonic()-started)
                    scores.append(score); print(f"  {name}: AUC={score['roc_auc']:.4f}, AP={score['average_precision']:.4f}, Brier={score['brier']:.4f}",flush=True)
                    pred=df.loc[test,['id_registro','clase','provincia','year']].copy()
                    pred['spatial_group']=groups.loc[test].to_numpy(); pred['probability']=probability
                    pred['model']=name; pred['scheme']=scheme; pred['fold']=fold; pred['seed']=seed
                    predictions.append(pred)
                    pd.DataFrame(scores).to_csv(output_dir/'metrics.csv',index=False)
                if 'mlp' in cfg['models'] and 'attention' in cfg['models']:
                    frame=pd.concat(predictions[-len(cfg['models']):],ignore_index=True)
                    delta=paired_block_bootstrap(frame,reps=cfg['bootstrap_reps'],seed=seed)
                    deltas.append(dict(scheme=scheme,fold=fold,seed=seed,baseline='mlp',**delta))
                    if 'mlp_matched' in cfg['models']:
                        delta=paired_block_bootstrap(frame,baseline='mlp_matched',reps=cfg['bootstrap_reps'],seed=seed)
                        deltas.append(dict(scheme=scheme,fold=fold,seed=seed,baseline='mlp_matched',**delta))
                pd.DataFrame(split_rows).to_csv(output_dir/'split_audit.csv',index=False)
                pd.concat(predictions,ignore_index=True).to_csv(output_dir/'predictions.csv',index=False)
                pd.DataFrame(deltas).to_csv(output_dir/'attention_deltas.csv',index=False)
        metadata['status']='completed'
    except BaseException as exc:
        metadata['status']='failed'; metadata['error']=repr(exc); raise
    finally:
        (output_dir/'execution.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    return pd.DataFrame(scores)



### Gráficos de datos, desempeño, geografía y aprendizaje

#### Estilo, nombres y exportación

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from sklearn.metrics import roc_curve,roc_auc_score,precision_recall_curve,average_precision_score,confusion_matrix,brier_score_loss
from sklearn.calibration import calibration_curve

ORDER=['logistic','rf','histgb','mlp','mlp_matched','attention']
LABELS={'logistic':'Logistic regression','rf':'Random Forest','histgb':'Histogram boosting','mlp':'MLP','mlp_matched':'MLP (capacity control)','attention':'Feature attention'}
COLORS={'logistic':'#666666','rf':'#0072B2','histgb':'#009E73','mlp':'#CC79A7','mlp_matched':'#56B4E9','attention':'#D55E00'}
SCHEMES={'random_pairs':'Random pairs','spatial_blocks':'Spatial blocks','leave_province_out':'Province held out','temporal_retrospective':'Temporal transfer'}
FEATURE_LABELS={'elevacion':'Elevation (m)','pendiente':'Slope (degrees)','orientacion':'Aspect (degrees)','temperatura_C':'Temperature (°C)','precipitacion_mm':'Precipitation (mm)','NDVI':'NDVI','EVI':'EVI','NBR':'NBR'}

def paper_style():
    plt.rcParams.update({'font.family':'DejaVu Sans','font.size':9,'axes.titlesize':10,'axes.labelsize':9,'xtick.labelsize':8,'ytick.labelsize':8,
        'legend.fontsize':8,'axes.spines.top':False,'axes.spines.right':False,'pdf.fonttype':42,'ps.fonttype':42,'svg.fonttype':'none','savefig.facecolor':'white'})


def export_figure(fig,out,name,caption,manifest,provenance):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    if not getattr(fig,'_paper_layout_fixed',False):fig.tight_layout()
    files=[]
    for suffix in ['png','svg','pdf']:
        path=out/f'{name}.{suffix}'
        fig.savefig(path,dpi=600,bbox_inches='tight',pad_inches=.08)
        files.append(path.name)
    plt.close(fig)
    manifest.append(dict(figure=name,files=files,caption=caption,**provenance))
    (out/'figures_manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
    (out/'CAPTIONS.md').write_text('# Figure captions\n\n'+'\n\n'.join('**'+m['figure']+'** '+m['caption'] for m in manifest),encoding='utf-8')


def read_results(folder):
    folder=Path(folder)
    meta=json.loads((folder/'execution.json').read_text(encoding='utf-8'))
    if meta['status']!='completed':raise ValueError('La ejecución no está completada.')
    scores=pd.read_csv(folder/'metrics.csv',dtype={'fold':str})
    predictions=pd.read_csv(folder/'predictions.csv',dtype={'fold':str,'id_registro':str,'spatial_group':str})
    if predictions.duplicated(['scheme','fold','seed','model','id_registro']).any():raise ValueError('Predicciones duplicadas.')
    if not predictions.probability.between(0,1).all():raise ValueError('Scores fuera de rango.')
    return meta,scores,predictions


def choose_test(predictions,scheme,fold,seed):
    selected=predictions[(predictions.scheme==scheme)&(predictions.fold==str(fold))&(predictions.seed==seed)].copy()
    if selected.empty:raise ValueError('No existe la partición elegida.')
    ids=None
    for model in selected.model.unique():
        current=set(selected[selected.model==model].id_registro)
        if ids is not None and current!=ids:raise ValueError('Los modelos no comparten el mismo test.')
        ids=current
    if selected.groupby('id_registro').clase.nunique().max()!=1:raise ValueError('Etiquetas inconsistentes.')
    return selected



#### Galería calculada desde predicciones guardadas: no reentrena

In [ ]:
def build_paper_figures(results,out,dataset=None,inventory=None,reference_results=None,ablations=None,
                        scheme='temporal_retrospective',fold='2024',seed=42):
    paper_style();results=Path(results);out=Path(out);out.mkdir(parents=True,exist_ok=True)
    meta,scores,pred=read_results(results);selected=choose_test(pred,scheme,fold,seed)
    manifest=[]
    status='PILOT' if meta['config'].get('pilot') else 'VALIDATION'
    stamp=f'{status} | {SCHEMES.get(scheme,scheme)}: {fold} | seed {seed}'
    source=dict(data_sha256=meta['data_sha256'],run_status=status,scheme=scheme,fold=str(fold),seed=seed)
    models=[m for m in ORDER if m in selected.model.unique()]
    def save(fig,name,caption,extra=None):
        export_figure(fig,out,name,caption,manifest,dict(source,**(extra or {})))
    df=None
    if dataset:
        if hashlib.sha256(Path(dataset).read_bytes()).hexdigest()!=meta['data_sha256']:
            raise ValueError('El dataset no corresponde a esta ejecución: hash distinto.')
        df=pd.read_csv(dataset,dtype={'id_registro':str});df['date']=pd.to_datetime(df.date)

    # F01: localización de las muestras y partición geográfica.
    if df is not None:
        geographic=scores[scores.scheme=='spatial_blocks']
        if not geographic.empty:
            f=str(geographic.iloc[0]['fold']);s=int(geographic.iloc[0]['seed'])
            split=pd.read_csv(results/f'split_spatial_blocks_{f}_{s}.csv',dtype={'id_registro':str})
            plotting=df.merge(split[['id_registro','partition']],on='id_registro',how='left')
            fig,axes=plt.subplots(1,2,figsize=(7.2,4.4),sharex=True,sharey=True)
            for label,color in [(0,'#56B4E9'),(1,'#D55E00')]:
                part=plotting[plotting.clase==label]
                axes[0].scatter(part.longitud,part.latitud,s=1.4,color=color,alpha=.4,rasterized=True,label='Presence' if label else 'Pseudoabsence')
            for label,color in [('train','#0072B2'),('test','#D55E00')]:
                part=plotting[plotting.partition==label]
                axes[1].scatter(part.longitud,part.latitud,s=1.8,color=color,alpha=.45,rasterized=True,label=label.title())
            axes[0].set_title('(a) Available observations');axes[1].set_title(f'(b) Geographic partition: {f}')
            for ax in axes:
                ax.set_xlabel('Longitude (degrees)');ax.set_aspect(1/np.cos(np.deg2rad(plotting.latitud.mean())))
                ax.legend(markerscale=4,loc='best',frameon=False)
            axes[0].set_ylabel('Latitude (degrees)')
            save(fig,'F01_sampling_and_spatial_split','Sampling locations (all modeling years) and one geographic train/test partition from the development period. Points excluded by the spatial buffer are not part of the displayed train set. These are sampling points, not a susceptibility surface or administrative boundary map.',dict(scheme='spatial_blocks',fold=f,seed=s))

        # F02: el periodo histórico se mantiene explícitamente separado del dataset de modelado.
        if inventory:
            import unicodedata
            history=pd.read_csv(inventory)
            def clean(v):
                v=''.join(c for c in unicodedata.normalize('NFKD',str(v).lower()) if not unicodedata.combining(c)).strip(" '")
                return {'bolvar':'bolivar','caar':'canar'}.get(v,v)
            names={'azuay','bolivar','canar','carchi','chimborazo','cotopaxi','imbabura','loja','pichincha','tungurahua'}
            history['date']=pd.to_datetime(history.date,errors='coerce')
            history=history[history.provincia.map(clean).isin(names)&history.date.dt.year.between(2010,2025)].copy()
            description='Historical regional inventory, 2010–2025; recorded events before model-data cleaning'
        else:
            history=df[df.clase==1].copy();description='Modeling presences only, 2019–2025; not the complete historical inventory'
        history['year']=history.date.dt.year;history['month']=history.date.dt.month
        monthly=history.month.value_counts().reindex(range(1,13),fill_value=0).sort_index()
        heat=pd.crosstab(history.year,history.month).reindex(columns=range(1,13),fill_value=0).sort_index()
        fig,axes=plt.subplots(2,1,figsize=(7.2,6.0),gridspec_kw={'height_ratios':[1,1.5]})
        axes[0].bar(range(12),monthly,color=['#D55E00' if m in [8,9,10,11] else '#999999' for m in range(1,13)],width=.75)
        for i,value in enumerate(monthly):axes[0].text(i,value+monthly.max()*.015,f'{value:,}',ha='center',va='bottom',fontsize=6.5)
        axes[0].set_ylim(0,monthly.max()*1.15)
        axes[0].set(xticks=range(12),xticklabels=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'],ylabel='Recorded fires',title='(a) Monthly distribution')
        axes[0].grid(axis='y',alpha=.15)
        image=axes[1].imshow(heat.to_numpy(),aspect='auto',cmap='YlOrRd',origin='lower')
        axes[1].set(xticks=range(12),xticklabels=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'],yticks=range(len(heat)),yticklabels=heat.index,ylabel='Year',title='(b) Year–month distribution')
        fig.colorbar(image,ax=axes[1],label='Recorded fires',fraction=.025,pad=.02)
        save(fig,'F02_seasonality',description+'. Counts are descriptive and depend on inventory coverage; zero counts do not demonstrate an absence of fire.',dict(scheme='descriptive',fold=None,seed=None,run_status='DESCRIPTIVE',historical_inventory_sha256=hashlib.sha256(Path(inventory).read_bytes()).hexdigest() if inventory else None))
        heat.to_csv(out/'seasonality_year_month.csv')

        # F03: las colas se recortan solo para dibujar; nunca se altera el modelado.
        fig,axes=plt.subplots(2,4,figsize=(7.2,4.8))
        for ax,(feature,label) in zip(axes.flat,FEATURE_LABELS.items()):
            lo,hi=df[feature].quantile([.005,.995]);bins=np.linspace(lo,hi,28)
            for cls,color in [(0,'#56B4E9'),(1,'#D55E00')]:
                values=df.loc[df.clase==cls,feature];shown=values[values.between(lo,hi)]
                ax.hist(shown,bins=bins,density=True,histtype='step',linewidth=1.25,color=color,label='Presence' if cls else 'Pseudoabsence')
            ax.set_title(label,fontsize=9);ax.tick_params(axis='x',labelsize=7)
        axes[0,0].legend(frameon=False,fontsize=7)
        axes[0,0].set_ylabel('Density');axes[1,0].set_ylabel('Density')
        save(fig,'F03_feature_distributions','Predictor distributions by class. Each panel displays the central 99% of the pooled values for legibility; all observations are retained in model fitting. Precipitation-window provenance must be reconciled with the extraction version.',dict(scheme='descriptive',fold=None,seed=None,run_status='DESCRIPTIVE'))

    # F04: ROC y PR usan exactamente los mismos IDs de test entre modelos.
    fig,axes=plt.subplots(1,2,figsize=(7.2,3.5))
    for model in models:
        p=selected[selected.model==model];fpr,tpr,_=roc_curve(p.clase,p.probability);precision,recall,_=precision_recall_curve(p.clase,p.probability)
        axes[0].plot(fpr,tpr,color=COLORS[model],lw=1.3,label=f'{LABELS[model]}: {roc_auc_score(p.clase,p.probability):.3f}')
        axes[1].plot(recall,precision,color=COLORS[model],lw=1.3,label=f'{LABELS[model]}: {average_precision_score(p.clase,p.probability):.3f}')
    axes[0].plot([0,1],[0,1],ls='--',color='#999999',lw=.8)
    prevalence=selected[selected.model==models[0]].clase.mean();axes[1].axhline(prevalence,ls='--',color='#999999',lw=.8)
    axes[0].set(xlabel='False positive rate',ylabel='True positive rate',title='(a) ROC curves',xlim=(0,1),ylim=(0,1.01))
    axes[1].set(xlabel='Recall',ylabel='Precision',title='(b) Precision–recall curves',xlim=(0,1),ylim=(0,1.01))
    for ax in axes:ax.legend(frameon=False,fontsize=6.5,loc='lower left' if ax is axes[1] else 'lower right')
    fig.suptitle(stamp,fontsize=9);save(fig,'F04_roc_precision_recall',stamp+'. AUC and average precision are computed from the saved external-test predictions. PR baseline is the sampled test prevalence, not population prevalence.')

    # F05: frecuencias y proporciones por clase verdadera, con escala común.
    fig,axes=plt.subplots(2,3,figsize=(7.2,4.8));image=None
    for ax,model in zip(axes.flat,models):
        p=selected[selected.model==model];counts=confusion_matrix(p.clase,p.probability>=.5,labels=[0,1]);rates=counts/counts.sum(axis=1,keepdims=True)
        image=ax.imshow(rates,cmap='Blues',vmin=0,vmax=1)
        for (i,j),n in np.ndenumerate(counts):ax.text(j,i,f'{n:,}\n{rates[i,j]:.1%}',ha='center',va='center',fontsize=8,color='white' if rates[i,j]>.6 else 'black')
        ax.set(title=LABELS[model],xticks=[0,1],xticklabels=['PA','Fire'],yticks=[0,1],yticklabels=['PA','Fire'])
        ax.set_xlabel('Predicted label');ax.set_ylabel('Observed label');ax.tick_params(labelsize=7)
    for ax in list(axes.flat)[len(models):]:ax.axis('off')
    fig.suptitle(stamp,fontsize=9);save(fig,'F05_confusion_matrices',stamp+'. Confusion matrices at score threshold 0.5. PA denotes pseudoabsence and Fire denotes recorded presence. Counts and row-normalized percentages are shown on a common 0–100% scale.')

    # F06: calibración exclusivamente dentro del muestreo caso-control.
    fig,axes=plt.subplots(1,2,figsize=(7.2,3.5));briers=[]
    for model in models:
        p=selected[selected.model==model];observed,predicted=calibration_curve(p.clase,p.probability,n_bins=10,strategy='quantile')
        axes[0].plot(predicted,observed,'o-',color=COLORS[model],ms=3,lw=1,label=LABELS[model]);briers.append(brier_score_loss(p.clase,p.probability))
    axes[0].plot([0,1],[0,1],'--',color='#999999',lw=.8);axes[0].set(xlabel='Mean case-control score',ylabel='Observed presence fraction',title='(a) Sample reliability',xlim=(0,1),ylim=(0,1))
    axes[0].legend(frameon=False,fontsize=6.5)
    axes[1].barh([LABELS[m] for m in models],briers,color=[COLORS[m] for m in models]);axes[1].set(xlabel='Brier score (lower is better)',title='(b) Sample Brier score');axes[1].invert_yaxis()
    fig.suptitle(stamp,fontsize=9);save(fig,'F06_sample_calibration',stamp+'. Reliability curves use ten quantile bins. These diagnose calibration in the case-control sample; they do not establish real-world ignition probabilities.')

    # F07: provincias estables; nunca se promedian folds territoriales diferentes por su número.
    ref_meta,ref_scores,_=read_results(reference_results or results)
    if ref_meta['data_sha256']!=meta['data_sha256']:raise ValueError('La referencia usa otra versión del dataset.')
    province=ref_scores[ref_scores.scheme=='leave_province_out'].groupby(['fold','model']).roc_auc.mean().unstack()
    if not province.empty:
        columns=[m for m in ORDER if m in province]
        fig,ax=plt.subplots(figsize=(7.2,4.5));image=ax.imshow(province[columns],cmap='viridis',vmin=.5,vmax=1,aspect='auto')
        for i in range(len(province)):
            for j in range(len(columns)):
                value=province.iloc[i][columns[j]]
                if np.isfinite(value):ax.text(j,i,f'{value:.3f}',ha='center',va='center',color='white' if value<.8 else 'black',fontsize=8)
        ax.set(yticks=range(len(province)),yticklabels=province.index,xticks=range(len(columns)),xticklabels=[LABELS[m] for m in columns],title='Leave-one-province-out discrimination')
        fig.suptitle('PILOT' if ref_meta['config'].get('pilot') else 'VALIDATION',fontsize=9)
        if len(columns)>3:plt.setp(ax.get_xticklabels(),rotation=35,ha='right',fontsize=7)
        fig.colorbar(image,ax=ax,label='ROC-AUC',fraction=.03,pad=.03)
        save(fig,'F07_province_transfer','External-test AUC by held-out province, averaged over the available fitting seeds. All values are descriptive; repeated seeds do not create independent territories.',dict(scheme='leave_province_out',fold='all_available_provinces',seed=None,seeds=ref_meta['config'].get('seeds'),run_status='PILOT' if ref_meta['config'].get('pilot') else 'VALIDATION',reference_code_sha256=ref_meta.get('code_sha256')))

    # F08: curvas registradas en validación interna, no en el test externo.
    fig,axes=plt.subplots(2,2,figsize=(7.2,5.4));available=scores[['scheme','fold','seed']].drop_duplicates()
    panels=[]
    for s in SCHEMES:
        part=available[(available.scheme==s)&(available.seed==seed)]
        if not part.empty:panels.append((s,str(part.iloc[0]['fold'])))
    used=False
    for ax,(s,f) in zip(axes.flat,panels):
        for model in ['mlp','mlp_matched','attention']:
            file=results/f'learning_{s}_{f}_{seed}_{model}.csv'
            if file.exists():
                h=pd.read_csv(file);used=True
                ax.plot(h.epoch,h.validation_loss,color=COLORS[model],lw=1.3,label=LABELS[model]);ax.plot(h.epoch,h.training_loss,color=COLORS[model],ls=':',lw=.9)
        ax.set(title=f'{SCHEMES[s]}: {f}',xlabel='Epoch',ylabel='Binary cross-entropy');ax.grid(alpha=.15)
    for ax in list(axes.flat)[len(panels):]:ax.axis('off')
    if used:
        axes[0,0].legend(frameon=False,fontsize=6.5)
        fig.suptitle(f'{status} | Internal optimization trajectories',fontsize=9)
        save(fig,'F08_learning_curves','Internal training (dotted) and validation (solid) losses. External-test data are not used for early stopping. One available partition per design is displayed; these are not the final full-training refit trajectories.',dict(scheme='multiple_designs',fold='one_representative_per_design'))
    else:plt.close(fig)

    # F09: diferencias pareadas y sus intervalos aproximados, sin ocultar el cero.
    delta_path=results/'attention_deltas.csv'
    if delta_path.exists() and delta_path.stat().st_size>1:
        deltas=pd.read_csv(delta_path,dtype={'fold':str})
        if not deltas.empty:
            deltas['baseline']=deltas.get('baseline',pd.Series('mlp',index=deltas.index)).fillna('mlp')
            deltas=deltas[(deltas.seed==seed)&(deltas.status=='approximate_spatial_block_bootstrap')].copy()
            comparisons=list(deltas[['scheme','fold']].drop_duplicates().itertuples(index=False,name=None))
            def comparison_label(s,f):
                if s=='random_pairs':return 'Random pairs'
                if s=='spatial_blocks':return str(f).replace('fold_','Block ')
                if s=='leave_province_out':return 'Province: '+str(f)
                return 'Year: '+str(f)
            labels=[comparison_label(s,f) for s,f in comparisons]
            baselines=[b for b in ['mlp','mlp_matched'] if b in deltas.baseline.unique()]
            fig,axes=plt.subplots(1,len(baselines),figsize=(7.2,max(3.5,len(comparisons)*.27)),sharex=True,sharey=True,squeeze=False)
            for ax,baseline in zip(axes.flat,baselines):
                for i,(s,f) in enumerate(comparisons):
                    rows=deltas[(deltas.scheme==s)&(deltas.fold==f)&(deltas.baseline==baseline)]
                    if rows.empty:continue
                    row=rows.iloc[0];color=COLORS[baseline]
                    ax.hlines(i,row.lower,row.upper,color=color,lw=1.3)
                    ax.vlines([row.lower,row.upper],i-.07,i+.07,color=color,lw=.8)
                    ax.plot(row.delta_auc,i,'o',color=color,ms=3.5)
                ax.axvline(0,color='#777777',ls='--',lw=.8)
                ax.set(yticks=range(len(labels)),yticklabels=labels,xlabel='AUC difference',title='vs MLP' if baseline=='mlp' else 'vs capacity control')
                ax.tick_params(labelsize=7);ax.grid(axis='x',alpha=.15)
            axes[0,0].invert_yaxis();fig.suptitle(f'{status}: attention ablation | seed {seed}',fontsize=9)
            save(fig,'F09_attention_ablation','Paired AUC differences on the same external-test cases. Bars are approximate spatial-block bootstrap intervals from the recorded run. They are conditional on the sample and block design and are not a confirmatory claim of superiority.',dict(scheme='multiple_designs',fold='recorded_comparisons'))

    # F10: ablaciones reales alineadas por diseño, fold, semilla y modelo.
    if ablations:
        full=ref_scores[ref_scores.seed==seed].copy();keys=['scheme','fold','seed','model']
        records=[]
        for name,folder in ablations.items():
            ameta,ascores,_=read_results(folder)
            if ameta['data_sha256']!=meta['data_sha256']:raise ValueError('Ablación de otra versión del dataset.')
            merged=ascores.merge(full,on=keys,suffixes=('_variant','_full'),validate='one_to_one')
            for (s,m),part in merged.groupby(['scheme','model']):records.append(dict(variant=name,scheme=s,model=m,delta=float((part.roc_auc_variant-part.roc_auc_full).mean())))
        table=pd.DataFrame(records)
        short_schemes={'random_pairs':'Random pairs','spatial_blocks':'Spatial blocks','leave_province_out':'Province','temporal_retrospective':'Temporal'}
        short_models={'logistic':'Logistic','rf':'RF','histgb':'Boosting'}
        table['row']=table.scheme.map(short_schemes)+' | '+table.model.map(short_models)
        heat=table.pivot(index='row',columns='variant',values='delta')
        limit=max(.005,float(np.nanmax(np.abs(heat.to_numpy()))))
        fig,ax=plt.subplots(figsize=(7.2,max(4,len(heat)*.28)));image=ax.imshow(heat,aspect='auto',cmap='RdBu',norm=TwoSlopeNorm(vmin=-limit,vcenter=0,vmax=limit))
        for (i,j),value in np.ndenumerate(heat.to_numpy()):
            if np.isfinite(value):ax.text(j,i,f'{value:+.3f}',ha='center',va='center',fontsize=7,color='white' if abs(value)>.65*limit else 'black')
        variant_labels={'no_climate':'Without climate','no_spectral':'Without spectral\nindices','raw_aspect':'Raw aspect'}
        ax.set(yticks=range(len(heat)),yticklabels=heat.index,xticks=range(len(heat.columns)),xticklabels=[variant_labels.get(c,c) for c in heat.columns],title='Covariate sensitivity: change relative to full features')
        fig.suptitle(('PILOT' if ref_meta['config'].get('pilot') else 'VALIDATION')+f' | seed {seed}',fontsize=9)
        fig.colorbar(image,ax=ax,label='Change in ROC-AUC',fraction=.03,pad=.03)
        save(fig,'F10_covariate_ablation','Mean paired differences from the full-feature model for matching external partitions and fitting seed. Negative values indicate lower AUC after the change. These are descriptive differences, not independent-fold confidence intervals.',dict(scheme='multiple_designs',fold='matching_external_partitions',run_status='PILOT' if ref_meta['config'].get('pilot') else 'VALIDATION',reference_code_sha256=ref_meta.get('code_sha256')))
        table.to_csv(out/'covariate_ablation_values.csv',index=False)
    return manifest


### Recuperación comprobada de modelos y explicabilidad

#### Recuperación comprobada del modelo y muestras de explicación

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

def explain_paper_figures(results,out,dataset,api=None,n_shap=120,n_background=30,coalitions=256,
                          permutation_n=1000,permutation_repeats=5,scheme='temporal_retrospective',fold='2024',seed=42):
    import shap
    import torch
    if api is None:
        import research_validation
        api=vars(research_validation)
    # En el cuaderno estas funciones están visibles en las celdas anteriores.
    if 'export_figure' not in globals():
        from paper_figures import export_figure,FEATURE_LABELS,COLORS,LABELS,paper_style,read_results,choose_test
    else:
        export_figure=globals()['export_figure'];FEATURE_LABELS=globals()['FEATURE_LABELS'];COLORS=globals()['COLORS'];LABELS=globals()['LABELS'];paper_style=globals()['paper_style'];read_results=globals()['read_results'];choose_test=globals()['choose_test']
    paper_style();results=Path(results);out=Path(out)
    meta,scores,predictions=read_results(results);cfg=meta['config'].copy();cfg['seed']=seed
    if cfg['feature_set']!='complete':raise ValueError('SHAP comparativo requiere la configuración completa de ocho variables físicas.')
    import hashlib
    if hashlib.sha256(Path(dataset).read_bytes()).hexdigest()!=meta['data_sha256']:raise ValueError('Hash de dataset distinto.')
    df=api['load_data'](dataset);features=api['FEATURES'];matrix=api['design_matrix'](df,cfg['feature_set']);X=matrix.to_numpy(dtype=np.float32);y=df.clase.to_numpy()
    assignment=pd.read_csv(results/f'split_{scheme}_{fold}_{seed}.csv',dtype={'id_registro':str})
    lookup=pd.Series(df.index,index=df.id_registro.astype(str))
    train=lookup.loc[assignment.loc[assignment.partition=='train','id_registro']].to_numpy()
    test=lookup.loc[assignment.loc[assignment.partition=='test','id_registro']].to_numpy()
    recorded=choose_test(predictions,scheme,fold,seed)
    if not {'rf','attention'}.issubset(set(recorded.model)):
        raise ValueError('Esta explicación comparativa requiere RF y attention en la corrida seleccionada.')
    rf=api['classic_model']('rf',cfg);rf.fit(X[train],y[train])
    fitted={}
    def keep(model,scaler):fitted.update(model=model,scaler=scaler)
    probability,training=api['fit_neural'](X,y,df,train,test,scheme,True,cfg,on_fitted=keep)
    def transform(raw):
        return api['design_matrix'](pd.DataFrame(np.asarray(raw),columns=features),cfg['feature_set']).to_numpy(dtype=np.float32)
    def predict_rf(raw):return rf.predict_proba(transform(raw))[:,1]
    def predict_attention(raw):
        values=fitted['scaler'].transform(transform(raw));parts=[]
        with torch.no_grad():
            for start in range(0,len(values),2048):
                tensor=torch.tensor(values[start:start+2048],dtype=torch.float32)
                parts.append(torch.sigmoid(fitted['model'](tensor)).numpy())
        return np.concatenate(parts)
    raw=df[features].to_numpy()
    predictors={'rf':predict_rf,'attention':predict_attention}
    max_errors={}
    for name,predictor in predictors.items():
        recalculated=pd.DataFrame({'id_registro':df.loc[test,'id_registro'].to_numpy(),'recalculated':predictor(raw[test])})
        joined=recorded[recorded.model==name].merge(recalculated,on='id_registro',validate='one_to_one')
        if len(joined)!=len(test):raise ValueError('Faltan predicciones para comprobar el ajuste.')
        error=float(np.max(np.abs(joined.probability-joined.recalculated)));max_errors[name]=error
        if error>1e-5:raise ValueError(f'No se reprodujo el modelo {name}: error {error:.6g}. Usa el entorno original o sus checkpoints; no se generarán explicaciones de otro ajuste.')
    rng=np.random.default_rng(seed)
    test_sample=rng.choice(test,min(n_shap,len(test)),replace=False)
    background=rng.choice(train,min(n_background,len(train)),replace=False)
    permutation_sample=rng.choice(test,min(permutation_n,len(test)),replace=False)
    shap_values={};permutations={};additivity={}
    manifest_path=out/'figures_manifest.json';manifest=json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.exists() else []
    provenance=dict(data_sha256=meta['data_sha256'],run_status='PILOT' if cfg.get('pilot') else 'VALIDATION',scheme=scheme,fold=str(fold),seed=seed,
        reproduction_max_abs_error=max_errors,n_explained=len(test_sample),background_n=len(background),coalitions=coalitions,
        permutation_cases=len(permutation_sample),permutation_repeats=permutation_repeats,
        predictions_sha256=hashlib.sha256((results/'predictions.csv').read_bytes()).hexdigest(),explainer_version=shap.__version__,
        explanation_code_sha256=globals().get('EXPLANATION_CODE_SHA','unversioned'))
    for name,predictor in predictors.items():
        print('Calculando SHAP:',name,flush=True)
        # Mismo explainer, background y casos; orientación se transforma conjuntamente a seno/coseno.
        np.random.seed(seed)
        explainer=shap.KernelExplainer(predictor,raw[background],link='identity')
        values=np.asarray(explainer.shap_values(raw[test_sample],nsamples=coalitions,l1_reg=0.0,silent=True))
        if values.ndim==3:values=values[:,:,0]
        if values.shape!=(len(test_sample),len(features)):raise ValueError('Forma SHAP inesperada.')
        shap_values[name]=values
        additivity[name]=float(np.max(np.abs(values.sum(axis=1)+float(np.asarray(explainer.expected_value).ravel()[0])-predictor(raw[test_sample]))))
        if additivity[name]>1e-3:raise ValueError('Falla la comprobación aditiva de SHAP.')
        pd.DataFrame(values,columns=features).assign(id_registro=df.loc[test_sample,'id_registro'].to_numpy()).to_csv(out/f'shap_values_{name}.csv',index=False)
        base=roc_auc_score(y[permutation_sample],predictor(raw[permutation_sample]));drops=np.empty((len(features),permutation_repeats))
        for j in range(len(features)):
            for repeat in range(permutation_repeats):
                shuffled=raw[permutation_sample].copy();order=np.random.default_rng(seed+j*100+repeat).permutation(len(shuffled));shuffled[:,j]=shuffled[order,j]
                drops[j,repeat]=base-roc_auc_score(y[permutation_sample],predictor(shuffled))
        permutations[name]=drops

    with torch.no_grad():
        scaled=fitted['scaler'].transform(X[test_sample]);tensor=torch.tensor(scaled,dtype=torch.float32);model=fitted['model']
        tokens=tensor.unsqueeze(-1)*model.scale.unsqueeze(0)+model.bias.unsqueeze(0)
        _,weights=model.attention(tokens,tokens,tokens,need_weights=True,average_attn_weights=False)
        routing=weights.mean(dim=(0,1)).numpy()
    np.testing.assert_allclose(routing.sum(axis=1),1,atol=1e-5)
    np.savez_compressed(out/'explanation_plot_data.npz',raw_samples=raw[test_sample],shap_rf=shap_values['rf'],shap_attention=shap_values['attention'],
        permutation_rf=permutations['rf'],permutation_attention=permutations['attention'],routing=routing,features=np.asarray(features,dtype=str),matrix_columns=np.asarray(matrix.columns,dtype=str))
    render_meta=dict(provenance=provenance,max_errors=max_errors,additivity=additivity,selected_epochs=training['selected_epochs'])
    (out/'explanation_render_metadata.json').write_text(json.dumps(render_meta,indent=2),encoding='utf-8')
    return render_explanation_cache(out)



#### Redibujar explicabilidad sin ajustar modelos ni recalcular SHAP

In [ ]:
def render_explanation_cache(out):
    """Modifica presentación a partir de los valores originales guardados."""
    if 'export_figure' not in globals():
        from paper_figures import export_figure,FEATURE_LABELS,COLORS,LABELS,paper_style
    else:
        export_figure=globals()['export_figure'];FEATURE_LABELS=globals()['FEATURE_LABELS'];COLORS=globals()['COLORS'];LABELS=globals()['LABELS'];paper_style=globals()['paper_style']
    out=Path(out);paper_style()
    meta=json.loads((out/'explanation_render_metadata.json').read_text(encoding='utf-8'))
    provenance=meta['provenance'];max_errors=meta['max_errors'];additivity=meta['additivity'];training=dict(selected_epochs=meta['selected_epochs'])
    cfg=dict(pilot=provenance['run_status']=='PILOT');scheme=provenance['scheme'];fold=provenance['fold']
    with np.load(out/'explanation_plot_data.npz',allow_pickle=False) as cache:
        raw=cache['raw_samples'].copy();features=cache['features'].tolist();test_sample=np.arange(len(raw))
        shap_values={name:cache['shap_'+name].copy() for name in ['rf','attention']}
        permutations={name:cache['permutation_'+name].copy() for name in ['rf','attention']}
        routing=cache['routing'].copy();matrix=pd.DataFrame(columns=cache['matrix_columns'].tolist())
    predictors={'rf':None,'attention':None}
    manifest=json.loads((out/'figures_manifest.json').read_text(encoding='utf-8'))
    manifest=[m for m in manifest if not m['figure'].startswith(('F11_','F12_','F13_'))]
    # F11: orden y escala comunes; colores enlazados a las mismas filas explicadas.
    importance=np.mean([np.abs(values).mean(axis=0) for values in shap_values.values()],axis=0)
    order=np.argsort(importance)
    fig,axes=plt.subplots(1,2,figsize=(7.2,4.4),sharex=True,sharey=True)
    stamp=f"{'PILOT' if cfg.get('pilot') else 'VALIDATION'} | {scheme.replace('_',' ')}: {fold} | {len(test_sample)} explained cases"
    fig.suptitle(stamp,fontsize=8.5)
    limit=max(np.max(np.abs(v)) for v in shap_values.values())*1.06
    cmap=plt.get_cmap('coolwarm')
    for ax,(name,values) in zip(axes,shap_values.items()):
        for position,j in enumerate(order):
            v=values[:,j];x=raw[test_sample,j];lo,hi=np.quantile(x,[.01,.99]);normalized=np.clip((x-lo)/(hi-lo if hi>lo else 1),0,1)
            edges=np.linspace(float(v.min())-1e-8,float(v.max())+1e-8,25);bins=np.digitize(v,edges);offset=np.zeros(len(v))
            for b in np.unique(bins):
                idx=np.flatnonzero(bins==b);offset[idx]=np.linspace(-.25,.25,len(idx)) if len(idx)>1 else 0
            ax.scatter(v,position+offset,c=normalized,cmap=cmap,vmin=0,vmax=1,s=9,alpha=.8,edgecolors='none',rasterized=True)
        ax.axvline(0,color='#888888',lw=.7);ax.set(title=LABELS[name],xlabel='SHAP contribution to case-control score',xlim=(-limit,limit),yticks=range(len(features)),yticklabels=[FEATURE_LABELS[features[j]] for j in order]);ax.grid(axis='y',alpha=.1)
    # Barra de color separada para no reducir un panel respecto al otro.
    scalar=plt.cm.ScalarMappable(norm=plt.Normalize(0,1),cmap=cmap)
    colorbar=fig.colorbar(scalar,ax=axes.tolist(),fraction=.025,pad=.03);colorbar.set_ticks([0,1],labels=['Low','High']);colorbar.set_label('Relative feature value')
    fig.subplots_adjust(right=.87,wspace=.1)
    # El exportador usa tight_layout; la barra se coloca como eje explícito después.
    fig.delaxes(colorbar.ax)
    fig.tight_layout(rect=[0,0,.93,1]);cax=fig.add_axes([.94,.2,.014,.6]);fig.colorbar(scalar,cax=cax,ticks=[0,1]).ax.set_yticklabels(['Low','High'])
    fig._paper_layout_fixed=True
    export_figure(fig,out,'F11_shap_comparison','SHAP distributions for reproduced RF and feature-attention fits on the same external-test sample and training-only reference background. Both use KernelExplainer on the eight physical variables; the orientation sine/cosine transformation is inside the prediction function. A common order, x scale and color mapping are used. These are model associations, not causal effects.',manifest,provenance)

    # F12: dos indicadores con unidades distintas; no se normalizan artificialmente al 100%.
    fig,axes=plt.subplots(1,2,figsize=(7.2,4.3),sharey=True)
    fig.suptitle(stamp,fontsize=8.5)
    positions=np.arange(len(features));width=.32
    for k,name in enumerate(['rf','attention']):
        offset=(k-.5)*width
        axes[0].barh(positions+offset,np.abs(shap_values[name]).mean(axis=0)[order],height=width,color=COLORS[name],label=LABELS[name])
        means=permutations[name].mean(axis=1)[order];std=permutations[name].std(axis=1)[order]
        axes[1].barh(positions+offset,means,height=width,color=COLORS[name],xerr=std,error_kw={'elinewidth':.7,'capsize':2})
    axes[0].set(yticks=positions,yticklabels=[FEATURE_LABELS[features[j]] for j in order],xlabel='Mean |SHAP| (score units)',title='(a) Global attribution')
    axes[1].set(xlabel='ROC-AUC reduction after permutation',title='(b) Permutation importance');axes[1].axvline(0,color='gray',lw=.7)
    axes[0].legend(frameon=False,fontsize=7)
    export_figure(fig,out,'F12_global_attribution_permutation','Global mean absolute SHAP and external-test permutation importance. Permutation bars show mean and standard deviation across repeated permutations, not confidence intervals. Correlated predictors can share or mask importance; the two panels have different units.',manifest,provenance)

    # F13: matriz real de routing de atención, no importancia causal de variables.
    # Matriz real recuperada del caché numérico del ajuste comprobado.
    names=[FEATURE_LABELS.get(n,{'orientacion_sin':'Aspect: sine','orientacion_cos':'Aspect: cosine'}.get(n,n)) for n in matrix.columns]
    fig,ax=plt.subplots(figsize=(7.2,5.4));image=ax.imshow(routing,cmap='Blues',vmin=0,vmax=max(.2,float(routing.max())))
    fig.suptitle(stamp,fontsize=8.5)
    ax.set(xticks=range(len(names)),xticklabels=names,yticks=range(len(names)),yticklabels=names,xlabel='Key token',ylabel='Query token',title='Mean attention routing between input tokens')
    plt.setp(ax.get_xticklabels(),rotation=55,ha='right')
    for (i,j),v in np.ndenumerate(routing):ax.text(j,i,f'{v:.2f}',ha='center',va='center',fontsize=6.5,color='white' if v>image.norm.vmax*.6 else 'black')
    fig.colorbar(image,ax=ax,label='Mean attention weight',fraction=.035,pad=.03)
    export_figure(fig,out,'F13_attention_routing','Attention routing averaged over heads and explained test samples. Query rows sum to one. Orientation is represented by two derived tokens, so the matrix has nine channels for eight physical variables. These routing weights are not SHAP, permutation importance or causal influence.',manifest,provenance)
    summary=[]
    for name in predictors:
        for j,feature in enumerate(features):summary.append(dict(model=name,feature=feature,mean_abs_shap=float(np.abs(shap_values[name][:,j]).mean()),permutation_auc_drop=float(permutations[name][j].mean()),permutation_sd=float(permutations[name][j].std())))
    pd.DataFrame(summary).to_csv(out/'explainability_summary.csv',index=False)
    pd.DataFrame(routing,index=matrix.columns,columns=matrix.columns).to_csv(out/'attention_routing.csv')
    (out/'explanation_checks.json').write_text(json.dumps(dict(reproduction_max_abs_error=max_errors,shap_additivity_error=additivity,selected_epochs=training['selected_epochs'],**{k:v for k,v in provenance.items() if k not in ['reproduction_max_abs_error']}),indent=2),encoding='utf-8')
    return manifest


### Conexión de todas las etapas y guía de manuscrito

#### Configuración del flujo completo

In [ ]:
from pathlib import Path
import contextlib
import copy
import hashlib
import json
import sys
import importlib.metadata
import inspect
import types
import pandas as pd

MASTER_DEFAULTS=dict(mode='COMPLETO',force=False,run_ablations=True,run_explanations=True,
    epochs=40,patience=8,seeds=[42,137,271],rf_trees=300,
    block_km=20,buffer_km=5,grid_offset_km=0,n_splits=5,
    ablation_models=['logistic','rf','histgb'],ablation_seeds=[42],
    explanation_cases=300,background_cases=50,coalitions=256,
    permutation_cases=1000,permutation_repeats=5,
    source_institution='',source_version='',extraction_status='Pendiente: reconciliar bitácora y script GEE')

class WorkflowLog:
    def __init__(self,*streams):self.streams=streams
    def write(self,text):
        for stream in self.streams:stream.write(text);stream.flush()
        return len(text)
    def flush(self):
        for stream in self.streams:stream.flush()


def callable_fingerprint(function):
    """Detecta ediciones de una función sin depender del nombre temporal de la celda."""
    try:
        content=inspect.getsource(function).replace('\r\n','\n')
    except (OSError,TypeError):
        def normalized(code):
            constants=[]
            for value in code.co_consts:
                constants.append(normalized(value) if isinstance(value,types.CodeType) else repr(value))
            return dict(bytecode=code.co_code.hex(),constants=constants,names=code.co_names,
                variables=code.co_varnames,argcount=code.co_argcount,kwonly=code.co_kwonlyargcount,
                freevars=code.co_freevars,cellvars=code.co_cellvars)
        content=json.dumps(normalized(function.__code__),sort_keys=True)
    return hashlib.sha256(content.encode()).hexdigest()


def runtime_fingerprint(api):
    names=['load_data','design_matrix','spatial_groups','nearest_m','purge','make_splits',
        'inner_validation','classic_model','neural_model','fit_neural','metrics','paired_block_bootstrap','run_validation']
    values=dict(functions={name:callable_fingerprint(api[name]) for name in names},
        features=api['FEATURES'],defaults=api['DEFAULT'])
    return hashlib.sha256(json.dumps(values,sort_keys=True).encode()).hexdigest(),values


def stage_notice(stage,message):
    print('\n'+'='*72+'\n'+stage+'\n'+message+'\n'+'='*72,flush=True)


def master_config(settings,api):
    cfg=copy.deepcopy(api['DEFAULT'])
    for name in ['epochs','patience','seeds','rf_trees','block_km','buffer_km','grid_offset_km','n_splits']:
        cfg[name]=copy.deepcopy(settings[name])
    if settings['mode']=='RAPIDO':
        cfg.update(pilot=True,seeds=[42],epochs=min(cfg['epochs'],8),patience=min(cfg['patience'],3),rf_trees=min(cfg['rf_trees'],100),bootstrap_reps=100)
    elif settings['mode']=='COMPLETO':cfg['pilot']=False
    else:raise ValueError('Elige COMPLETO o RAPIDO.')
    return cfg


def reuse_or_run(dataset,folder,config,api,code_hash,force=False):
    """Reutiliza únicamente una ejecución íntegra de los mismos datos/código/configuración."""
    folder=Path(folder);meta_file=folder/'execution.json'
    desired=copy.deepcopy(api['DEFAULT']);desired.update(config)
    digest=hashlib.sha256(Path(dataset).read_bytes()).hexdigest()
    if not force and meta_file.exists():
        previous=json.loads(meta_file.read_text(encoding='utf-8'))
        files=['metrics.csv','predictions.csv','split_audit.csv']
        versions={p:importlib.metadata.version(p) for p in ['numpy','pandas','scipy','scikit-learn','torch','pyproj']}
        if previous.get('status')=='completed' and previous.get('config')==desired and previous.get('data_sha256')==digest and previous.get('code_sha256')==code_hash and previous.get('versions')==versions and previous.get('python')==sys.version and all((folder/name).is_file() for name in files):
            print('Etapa ya completada y compatible; se reutiliza:',folder.name,flush=True)
            return pd.read_csv(folder/'metrics.csv')
    folder.mkdir(parents=True,exist_ok=True)
    with (folder/'console.log').open('w',encoding='utf-8') as log:
        with contextlib.redirect_stdout(WorkflowLog(sys.stdout,log)):
            return api['run_validation'](dataset,folder,config)


def table_markdown(frame):
    cols=[str(c) for c in frame.columns]
    lines=['| '+' | '.join(cols)+' |','|'+'|'.join(['---']*len(cols))+'|']
    for row in frame.itertuples(index=False,name=None):
        values=[]
        for value in row:
            if isinstance(value,float):values.append('—' if pd.isna(value) else f'{value:.4f}')
            else:values.append(str(value).replace('|','/'))
        lines.append('| '+' | '.join(values)+' |')
    return '\n'.join(lines)



#### Informe orientado al manuscrito, basado solo en resultados disponibles

In [ ]:
def write_paper_guide(folder,df,scores,audit,settings,config,figure_manifest):
    folder=Path(folder)
    summary=scores.groupby(['scheme','model']).agg(AUC=('roc_auc','mean'),AP=('average_precision','mean'),Brier=('brier','mean'),evaluaciones=('roc_auc','size')).reset_index()
    summary.to_csv(folder/'resumen_modelos.csv',index=False)
    province=scores[scores.scheme=='leave_province_out'].groupby(['fold','model'])[['roc_auc','average_precision']].mean().reset_index()
    province.to_csv(folder/'resumen_provincias.csv',index=False)
    status='PILOTO TÉCNICO; no resultados definitivos' if config['pilot'] else 'Ejecución del protocolo completo configurado; requiere revisión científica'
    count=df.clase.value_counts().to_dict()
    figures='\n'.join('- **'+m['figure']+'**: '+m['caption'] for m in figure_manifest)
    source_checks=audit.get('source_checks')
    source_status='Contrastadas las tablas fuente aportadas; revisar el JSON de auditoría.' if isinstance(source_checks,dict) else 'No se aportaron todas las tablas fuente: no se verificaron contra ellas el buffer ni los controles de Sentinel-2.'
    guide=f'''# Guía de redacción del paper

Estado de esta ejecución: **{status}**.

## Qué datos y versión se analizaron

Observaciones: {len(df):,}. Presencias: {count.get(1,0):,}. Pseudoausencias: {count.get(0,0):,}.
Periodo del CSV: {df.date.min().date()} a {df.date.max().date()}.
Las coordenadas recurrentes en años distintos no deben tratarse como eventos independientes sin control geográfico.

Fuente institucional declarada: {settings['source_institution'] or 'PENDIENTE DE DOCUMENTAR'}.
Versión del inventario: {settings['source_version'] or 'PENDIENTE DE DOCUMENTAR'}.
Estado de extracción: {settings['extraction_status']}.
{source_status}

## Introducción

Explica el problema de transferencia entre territorios y periodos y formula una
pregunta verificable sobre el valor añadido de la atención. La arquitectura por sí
sola no demuestra novedad. La revisión del estado del arte y la revista objetivo
siguen siendo decisiones del manuscrito, no resultados que este código invente.

## Métodos que corresponden a esta ejecución

Se usaron ocho covariables físicas; orientación circular genera nueve canales
en la configuración completa. ID, fecha, provincia y coordenadas se conservan
para trazabilidad y particiones, no como predictores del modelo.

Desarrollo hasta {config['development_end']}; bloques nominales de {config['block_km']} km
en EPSG:6933; buffer entre entrenamiento y test de {config['buffer_km']} km esféricos;
test temporal {config['temporal_years']} con embargo de 90 días. Semillas: {config['seeds']}.
2025 no se evalúa por defecto. Su independencia histórica todavía debe comprobarse.

Los escaladores se ajustan solo sobre entrenamiento. Las épocas neuronales se
seleccionan con validación interna y luego se reajusta con el entrenamiento externo
completo. Se fijan hiperparámetros; esta implementación no hace búsqueda con CV anidada.
Los controles incluyen MLP sin atención y MLP con presupuesto de parámetros similar.

La exclusión global de pseudoausencias basada en incendios conocidos de todo el
periodo hace que el contraste temporal sea retrospectivo. No debe describirse
como un pronóstico operacional. Tampoco debe afirmarse calibración poblacional:
se trata de un muestreo caso-control.

## Resultados disponibles

Los promedios siguientes son descriptivos. Folds y semillas no son réplicas
independientes para construir errores estándar o pruebas de significación.

{table_markdown(summary)}

Examina `resumen_provincias.csv` antes de generalizar un promedio nacional.
Compara los tipos de error, PR y Brier además de ROC-AUC.
Las diferencias pareadas de atención y sus intervalos están en
`principal/attention_deltas.csv`: usa exactamente el mismo test y no selecciones
solo los folds favorables. Un intervalo que incluya cero no sostiene una ventaja
clara en ese contraste.

## Ablaciones y explicabilidad

Ablaciones ejecutadas: {settings['run_ablations']}. Modelos de esas ablaciones:
{settings['ablation_models']}; semillas: {settings['ablation_seeds']}.
No atribuyas resultados de los modelos clásicos a redes que no se hayan evaluado
en esas ablaciones. Las campañas nuevas de pseudoausencias no se simulan en este
flujo: requieren el código GEE exacto y extracciones independientes comparables.

Explicabilidad solicitada: {settings['run_explanations']}. SHAP y permutación
describen el modelo y dependen del background, correlaciones y muestra. Los
pesos de atención son routing entre tokens, no efectos causales ni SHAP.
Comprueba `figuras/explanation_checks.json` si se generó esta etapa.

## Figuras y su función en el argumento

{figures}

F01 muestra puntos y particiones, no una superficie de susceptibilidad. Para un
mapa continuo deben prepararse predictores espaciales compatibles, dominio de
aplicación, soporte temporal y extrapolación. Los mapas históricos no se importan
como si pertenecieran automáticamente a este ajuste.

## Discusión y conclusiones

Relaciona los resultados territoriales con cobertura ambiental y limitaciones
de muestreo antes de proponer mecanismos ecológicos. Conserva resultados
negativos: la atención puede no mejorar al control de capacidad.
No reutilices cifras, importancias o interpretaciones de la versión antigua
del dataset en las figuras nuevas sin contrastar su procedencia.

## Pendientes antes del envío

- Fuente y versión oficiales del inventario, permisos y acceso a datos.
- Reconciliar clima de 30 días frente a 12 meses y exclusión de 2 km frente a 1–8 km
  entre la bitácora y el script aportado. Este cuaderno no modifica los CSV para ocultarlo.
- Verificar la adquisición preevento con el código de extracción exacto.
- Revisar sensibilidad de bloques/buffers y campañas reales de pseudoausencias.
- Comprobar un conjunto externo realmente independiente del trabajo histórico.
- Revisar estabilidad de SHAP y el dominio de los mapas antes de afirmar causalidad o utilidad operacional.

La ejecución reproducible organiza la evidencia; no garantiza aceptación Q1.
'''
    target=folder/'GUIA_PARA_REDACTAR_EL_PAPER.md';target.write_text(guide,encoding='utf-8')
    return target



#### Ejecución única: auditoría → validación → ablaciones → figuras → informe

In [ ]:
def complete_study(dataset,storage,api,settings=None,presence=None,pseudo=None,inventory=None,core_hash=None):
    cfg_settings=copy.deepcopy(MASTER_DEFAULTS);cfg_settings.update(settings or {})
    config=master_config(cfg_settings,api)
    df=api['load_data'](dataset)
    data_hash=hashlib.sha256(Path(dataset).read_bytes()).hexdigest()
    template_hash=api.get('NOTEBOOK_TEMPLATE_SHA',core_hash)
    core_hash,fingerprint=runtime_fingerprint(api)
    # La huella efectiva corresponde al código ejecutado, incluso si se editó una celda.
    api['NOTEBOOK_IMPLEMENTATION_SHA']=core_hash
    api['run_validation'].__globals__['NOTEBOOK_IMPLEMENTATION_SHA']=core_hash
    explanation_hash=callable_fingerprint(api['explain_paper_figures'])
    api['EXPLANATION_CODE_SHA']=explanation_hash
    api['explain_paper_figures'].__globals__['EXPLANATION_CODE_SHA']=explanation_hash
    identity=hashlib.sha256(json.dumps(dict(config=config,ablations=cfg_settings['ablation_models'],ablation_seeds=cfg_settings['ablation_seeds']),sort_keys=True).encode()).hexdigest()
    folder=Path(storage)/'estudios'/f'{data_hash[:10]}_{core_hash[:8]}_{identity[:8]}'
    folder.mkdir(parents=True,exist_ok=True)
    (folder/'runtime_code_fingerprint.json').write_text(json.dumps(dict(sha256=core_hash,template_sha256=template_hash,**fingerprint),indent=2),encoding='utf-8')
    environment={p:importlib.metadata.version(p) for p in ['numpy','pandas','scipy','scikit-learn','torch','pyproj','matplotlib','shap']}
    (folder/'environment.json').write_text(json.dumps(dict(python=sys.version,packages=environment),indent=2),encoding='utf-8')
    (folder/'requirements_ejecucion.txt').write_text('\n'.join(f'{name}=={version}' for name,version in environment.items())+'\n',encoding='utf-8')
    (folder/'configuration.json').write_text(json.dumps(dict(settings=cfg_settings,config=config,data_sha256=data_hash,core_sha256=core_hash),indent=2,ensure_ascii=False),encoding='utf-8')
    stage_notice('1/6 · Auditoría','Se comprueban datos y tablas fuente disponibles.')
    audit=api['run_audit'](dataset,folder/'auditoria',presence,pseudo,inventory)
    stage_notice('2/6 · Plan de validación','Se comprueban todas las particiones antes de entrenar.')
    planned=[]
    for seed in config['seeds']:
        for scheme,fold,train,test in api['make_splits'](df,dict(config,seed=seed)):
            planned.append(dict(scheme=scheme,fold=fold,seed=seed,train=len(train),test=len(test)))
    plan=pd.DataFrame(planned);plan.to_csv(folder/'plan_particiones.csv',index=False)
    print('Evaluaciones principales:',len(plan)*len(config['models']),flush=True)
    print(plan.to_string(index=False),flush=True)
    if not config['pilot']:print('Modo COMPLETO: puede tardar varias horas en CPU. Las etapas terminadas se conservan y reutilizan.',flush=True)
    stage_notice('3/6 · Comparación principal','Todos los modelos usan las mismas particiones externas.')
    scores=reuse_or_run(dataset,folder/'principal',config,api,core_hash,cfg_settings['force'])
    ablations={}
    if cfg_settings['run_ablations']:
        stage_notice('4/6 · Ablaciones','Se retira espectro o clima y se contrasta orientación cruda.')
        for feature_set in ['no_spectral','no_climate','raw_aspect']:
            variant=copy.deepcopy(config);variant.update(feature_set=feature_set,models=cfg_settings['ablation_models'],seeds=cfg_settings['ablation_seeds'])
            if config['pilot']:variant['seeds']=[42]
            destination=folder/('ablacion_'+feature_set)
            reuse_or_run(dataset,destination,variant,api,core_hash,cfg_settings['force'])
            ablations[feature_set]=destination
    stage_notice('5/6 · Galería de figuras','Se usan directamente las predicciones de las etapas anteriores.')
    figures=folder/'figuras'
    manifest=api['build_paper_figures'](folder/'principal',figures,dataset=dataset,inventory=inventory,
        reference_results=folder/'principal',ablations=ablations,seed=config['seeds'][0])
    if cfg_settings['run_explanations']:
        expected=dict(n_explained=min(cfg_settings['explanation_cases'],len(df[df.year==2024])),background_n=cfg_settings['background_cases'],coalitions=cfg_settings['coalitions'],
            permutation_cases=min(cfg_settings['permutation_cases'],len(df[df.year==2024])),permutation_repeats=cfg_settings['permutation_repeats'],
            predictions_sha256=hashlib.sha256((folder/'principal/predictions.csv').read_bytes()).hexdigest(),explainer_version=importlib.metadata.version('shap'),explanation_code_sha256=explanation_hash)
        cache=figures/'explanation_render_metadata.json';reuse=False
        if cache.exists() and (figures/'explanation_plot_data.npz').exists() and not cfg_settings['force']:
            previous=json.loads(cache.read_text(encoding='utf-8'))['provenance']
            reuse=all(previous.get(k)==v for k,v in expected.items()) and previous.get('data_sha256')==data_hash and previous.get('seed')==config['seeds'][0]
        if reuse:
            print('SHAP ya calculado: se redibuja desde el caché comprobado.',flush=True)
            manifest=api['render_explanation_cache'](figures)
        else:
            manifest=api['explain_paper_figures'](folder/'principal',figures,dataset,api=api,
                n_shap=cfg_settings['explanation_cases'],n_background=cfg_settings['background_cases'],coalitions=cfg_settings['coalitions'],
                permutation_n=cfg_settings['permutation_cases'],permutation_repeats=cfg_settings['permutation_repeats'],seed=config['seeds'][0])
    stage_notice('6/6 · Orientación para el manuscrito','Se genera una guía basada en los resultados realmente disponibles.')
    report=write_paper_guide(folder,df,scores,audit,cfg_settings,config,manifest)
    (folder/'workflow_completed.json').write_text(json.dumps(dict(status='completed',data_sha256=data_hash,core_sha256=core_hash,figures=len(manifest)),indent=2),encoding='utf-8')
    print('Estudio guardado en:',folder,'\nGuía para el paper:',report,flush=True)
    return dict(folder=folder,scores=scores,audit=audit,figures=manifest,report=report,config=config)


## D. Entrada única de datos

Carga el CSV de modelado. Si los tienes disponibles, puedes seleccionar también
en esta misma entrada las presencias fuente, pseudoausencias fuente e inventario
histórico. Son **datos originales**, no resultados de otros cuadernos.

- Obligatorio: el CSV con `id_registro`, fecha, provincia, latitud/longitud,
  ocho covariables y clase.
- Opcionales: `incendios_sierra_2019_2025_modelo.csv`,
  `pseudoausencias_sierra_2019_2025_final.csv`, `incendios_coordenadas_gee.csv`.

Sin el inventario histórico, la estacionalidad se calculará solo sobre las
presencias de modelado y se etiquetará así. Sin las tres tablas fuente no se
certificará el buffer ni el origen de las observaciones satelitales desde el CSV final.


In [ ]:
# Esta es la única solicitud de archivos de todo el cuaderno.
if FORZAR_CARGA_DATOS or not (DATOS/ARCHIVO_PRINCIPAL).is_file():
    from google.colab import files
    cargados=files.upload()
    for nombre,contenido in cargados.items():
        if Path(nombre).name!=nombre or not nombre.lower().endswith('.csv'):
            raise ValueError('Selecciona únicamente CSV de datos; no se necesitan ZIP.')
        (DATOS/nombre).write_bytes(contenido)
    del cargados

columnas_modelo=set(FEATURES+['id_registro','date','provincia','latitud','longitud','clase'])
candidatos=[];PRESENCIAS=None;PSEUDOAUSENCIAS=None;INVENTARIO=None
for ruta in sorted(DATOS.glob('*.csv')):
    columnas=set(pd.read_csv(ruta,nrows=0).columns)
    if columnas_modelo.issubset(columnas):candidatos.append(ruta)
    elif {'id_pseudo','id_presencia_referencia','latitud','longitud'}.issubset(columnas):PSEUDOAUSENCIAS=ruta
    elif {'id','n_observaciones_S2_validas','latitud','longitud'}.issubset(columnas):PRESENCIAS=ruta
    elif {'id','date','provincia','latitude','longitude'}.issubset(columnas):INVENTARIO=ruta

principal=DATOS/ARCHIVO_PRINCIPAL
if principal in candidatos:DATA=principal
elif len(candidatos)==1:DATA=candidatos[0]
else:raise ValueError('No se identificó un único dataset trazable. Revisa ARCHIVO_PRINCIPAL y las columnas; no renombres la versión antigua.')
datos=load_data(DATA)
print('Dataset:',DATA.name,'| filas:',len(datos),'| SHA-256:',hashlib.sha256(DATA.read_bytes()).hexdigest())
display(datos.head())
display(datos.groupby(['year','clase']).size().unstack(fill_value=0))
print('Tablas fuente disponibles:',{'presencias':PRESENCIAS is not None,'pseudoausencias':PSEUDOAUSENCIAS is not None,'inventario':INVENTARIO is not None})


## E. Ejecutar todo el estudio

Esta celda conecta las etapas: los modelos alimentan directamente las figuras;
las tablas y figuras alimentan la guía del paper. No hay que exportar, volver a
subir ni buscar resultados de otro cuaderno.

El protocolo completo compara seis modelos con tres semillas, cinco folds
geográficos, exclusión de cada provincia y transferencia temporal retrospectiva.
Las ablaciones configuradas comparan los tres modelos clásicos con una semilla;
no se atribuyen esos resultados a redes que no se hayan evaluado en ellas.

Los hiperparámetros están fijados. Las épocas se eligen con validación interna
del entrenamiento; no se optimiza contra el test externo. 2025 permanece bloqueado.
Si la sesión se interrumpe, vuelve a ejecutar el mismo cuaderno con los mismos
parámetros: reutiliza las etapas que hayan terminado y sean compatibles.


In [ ]:
AJUSTES=copy.deepcopy(MASTER_DEFAULTS)
AJUSTES.update(mode=MODO,force=RECALCULAR_ETAPAS,run_ablations=EJECUTAR_ABLACIONES,
    run_explanations=EJECUTAR_SHAP,source_institution=FUENTE_INSTITUCIONAL,
    source_version=VERSION_INVENTARIO,extraction_status=ESTADO_EXTRACCION)
# Parámetros científicos visibles. Cambiarlos crea una configuración distinta.
AJUSTES.update(block_km=20,buffer_km=5,grid_offset_km=0,n_splits=5,
    epochs=40,patience=8,seeds=[42,137,271],rf_trees=300,
    ablation_models=['logistic','rf','histgb'],ablation_seeds=[42],
    explanation_cases=300,background_cases=50,coalitions=256,
    permutation_cases=1000,permutation_repeats=5)
if MODO=='RAPIDO':
    AJUSTES.update(explanation_cases=120,background_cases=30)

display(AJUSTES)
ESTUDIO=complete_study(DATA,ALMACEN,globals(),AJUSTES,
    presence=PRESENCIAS,pseudo=PSEUDOAUSENCIAS,inventory=INVENTARIO,
    core_hash=NOTEBOOK_IMPLEMENTATION_SHA)


## F. Revisar resultados y figuras en este cuaderno

Estas salidas proceden de la ejecución anterior. Comprueba primero auditoría,
separación geográfica y diferencias por provincia. No concluyas superioridad de
la atención mirando solo un fold favorable. Las figuras conservan la etiqueta
PILOT si ejecutaste el modo rápido.


In [ ]:
display(ESTUDIO['audit'])
display(ESTUDIO['scores'].groupby(['scheme','model'])[['roc_auc','average_precision','brier']].agg(['mean','std']))
for figura in ESTUDIO['figures']:
    display(Markdown('### '+figura['figure']+'\n\n'+figura['caption']))
    display(Image(filename=str(ESTUDIO['folder']/'figuras'/(figura['figure']+'.png')),width=950))


## G. Orientación para redactar el paper

La guía siguiente usa los conteos, configuración, métricas y figuras realmente
generados. Distingue lo ejecutado de lo pendiente. Es una base de redacción,
no un texto que invente resultados o garantice aceptación editorial.

Incluye qué informar en introducción, métodos, resultados, discusión, figuras
principales y material suplementario. Revisa especialmente la procedencia del
inventario, la versión de extracción y la independencia del conjunto de prueba.


In [ ]:
display(Markdown(ESTUDIO['report'].read_text(encoding='utf-8')))
print('Todo el estudio se encuentra en:',ESTUDIO['folder'])


## H. Guardado final y colaboración

Con Drive activado ya quedan guardados datos, auditoría, logs, particiones,
métricas, predicciones, figuras y la guía del paper. No necesitas trasladarlos
a otro cuaderno. La descarga de una copia completa es opcional y solo ocurre al final.

Para colaborar en GitHub, propone cambios mediante una rama y pull request.
No guardes salidas del cuaderno con datos privados en un commit. El cuaderno
público se distribuye con todas las celdas de código visibles y sin resultados incrustados.


In [ ]:
DESCARGAR_COPIA_FINAL=False
if DESCARGAR_COPIA_FINAL:
    import shutil
    from google.colab import files
    copia=shutil.make_archive(str(ESTUDIO['folder'])+'_copia','zip',ESTUDIO['folder'])
    files.download(copia)
print('Trabajo guardado:',ESTUDIO['folder'])
